# CNN conduite — behavioral cloning du PID

Entraine un CNN leger a predire (steering, throttle) a partir de (masque 64x64 + 5 lidar).
Labels = sorties du PID baseline (dataset deja collecte, data/train + data/val).

Split par circuit (D10). Eval = MSE + simulation runtime sur circuits val.

Archi : `pilot.cnn_drive_arch.DriveCNN` (partage avec l'inference).

Output : `models/cnn_drive.pth`.

In [ ]:
import os, sys
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
print(f'Colab : {IS_COLAB}')

if IS_COLAB:
    if not Path('autonomous-twin').exists():
        !git clone --branch dev/ml-nohlan-lorenzo https://github.com/CharlesCar0105/autonomous-twin.git
    os.chdir('autonomous-twin')
    !pip -q install pygame pyzmq opencv-python pillow

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
print(f'ROOT : {ROOT}')

In [ ]:
import torch
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')

## Dataset : regenere si absent

In [ ]:
train_dir = ROOT / 'data' / 'train'
val_dir = ROOT / 'data' / 'val'

if not train_dir.exists() or not any(train_dir.iterdir()):
    !python scripts/gen_circuits.py --n 30 --seed 1000
    !python scripts/record_dataset.py --seconds 30 --frame-stride 3

print(f'train : {sum(1 for _ in train_dir.rglob("*.npz"))} frames')
print(f'val   : {sum(1 for _ in val_dir.rglob("*.npz"))} frames')

## Dataset PyTorch

Le masque GT stocke est en 128x128, on le downsize en 64x64 (nearest). Lidar normalise /300.
Augmentation : flip horizontal + inversion du signe de steering.

In [ ]:
import numpy as np
import random
from torch.utils.data import Dataset, DataLoader

STEER_MAX_DEG = 45.0
LIDAR_MAX_PX = 300.0
MASK_SIZE = 64


def _resize_nearest(a, size):
    h, w = a.shape
    ys = (np.arange(size) * h / size).astype(np.int64)
    xs = (np.arange(size) * w / size).astype(np.int64)
    return a[ys[:, None], xs[None, :]]


class DriveDataset(Dataset):
    def __init__(self, root: Path, augment: bool = False):
        self.files = sorted(root.rglob('*.npz'))
        self.augment = augment

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        d = np.load(self.files[idx])
        mask = _resize_nearest(d['mask'].astype(np.float32), MASK_SIZE)  # (64, 64)
        lidar = d['lidar'].astype(np.float32) / LIDAR_MAX_PX              # (5,)
        steer = np.float32(d['steering']) / STEER_MAX_DEG                # [-1, 1]
        throttle = np.float32(d['throttle'])                              # [0, 1]

        if self.augment and random.random() < 0.5:
            mask = mask[:, ::-1].copy()
            lidar = lidar[::-1].copy()
            steer = -steer

        return (
            torch.from_numpy(mask).unsqueeze(0),          # (1, 64, 64)
            torch.from_numpy(lidar),                       # (5,)
            torch.tensor([steer, throttle], dtype=torch.float32),  # (2,)
        )


train_ds = DriveDataset(train_dir, augment=True)
val_ds = DriveDataset(val_dir, augment=False)
print(f'train : {len(train_ds)}  val : {len(val_ds)}')

BATCH = 64
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

## Model + Training

In [ ]:
from pilot.cnn_drive_arch import DriveCNN

model = DriveCNN().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'parametres : {n_params:,}')

In [ ]:
import torch.nn as nn

EPOCHS = 40
LR = 1e-3

# Loss MSE avec poids : steering compte 2x throttle (plus critique en conduite).
def loss_fn(pred, target):
    err = (pred - target) ** 2
    return 2.0 * err[:, 0].mean() + err[:, 1].mean()

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=4)

best_val = float('inf')
history = {'train': [], 'val': [], 'steer_mae': [], 'throttle_mae': []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    tr = 0.0
    for mask, lidar, target in train_loader:
        mask, lidar, target = mask.to(DEVICE), lidar.to(DEVICE), target.to(DEVICE)
        optimizer.zero_grad()
        pred = model(mask, lidar)
        loss = loss_fn(pred, target)
        loss.backward()
        optimizer.step()
        tr += loss.item() * mask.size(0)
    tr /= len(train_ds)

    model.eval()
    vl = 0.0; mae_s = 0.0; mae_t = 0.0
    with torch.no_grad():
        for mask, lidar, target in val_loader:
            mask, lidar, target = mask.to(DEVICE), lidar.to(DEVICE), target.to(DEVICE)
            pred = model(mask, lidar)
            vl += loss_fn(pred, target).item() * mask.size(0)
            mae_s += (pred[:, 0] - target[:, 0]).abs().sum().item()
            mae_t += (pred[:, 1] - target[:, 1]).abs().sum().item()
    vl /= len(val_ds); mae_s /= len(val_ds); mae_t /= len(val_ds)

    scheduler.step(vl)
    history['train'].append(tr); history['val'].append(vl)
    history['steer_mae'].append(mae_s * STEER_MAX_DEG)  # en degres
    history['throttle_mae'].append(mae_t)

    mark = ''
    if vl < best_val:
        best_val = vl
        (ROOT / 'models').mkdir(exist_ok=True)
        torch.save(model.state_dict(), ROOT / 'models' / 'cnn_drive.pth')
        mark = ' *'
    print(f'ep {epoch:2d}  tr={tr:.5f}  vl={vl:.5f}  '
          f'steer_mae={mae_s*STEER_MAX_DEG:5.2f} deg  '
          f'thr_mae={mae_t:.3f}{mark}')

print(f'\nbest val loss : {best_val:.5f}')

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(history['train'], label='train'); axes[0].plot(history['val'], label='val')
axes[0].set_title('loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history['steer_mae']); axes[1].axhline(5, color='r', ls='--', alpha=0.5)
axes[1].set_title('steering MAE (deg)'); axes[1].grid(alpha=0.3)
axes[2].plot(history['throttle_mae']); axes[2].axhline(0.1, color='r', ls='--', alpha=0.5)
axes[2].set_title('throttle MAE'); axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Eval runtime : est-ce que le CNN tient la piste ?

MSE bas != conduite correcte (distribution shift). On fait rouler le CNN en boucle sur les circuits val.

In [ ]:
!python scripts/test_cnn.py --seconds 15

## Export

In [ ]:
path = ROOT / 'models' / 'cnn_drive.pth'
print(f'modele : {path}  ({path.stat().st_size / 1e6:.2f} Mo)')
if IS_COLAB:
    from google.colab import files
    files.download(str(path))